# Visualize linkage disequilibrium for SNPs
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import deque

import matplotlib as mpl
import numpy as np
import pandas as pd

from hk1_r12ter_analysis.config import GENE_ID, PROCESSED_DATA_DIR
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.linkage_disequilibrium import (
    create_LD_heatmap_w_chromosom_mapping,
)
from hk1_r12ter_analysis.util import ensure_iterable

# Set font family to Arial
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]

## Load data

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"

LD_method = "RH"
r2_thresh = 0

### Load data for SNP chromosome location

In [ ]:
index_col = "ID"
columns = ["RSID", "#CHROM", "POS"]
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_metadata = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=index_col
)

rsids = df_genomics_metadata.index.tolist()
df_genomics_metadata = df_genomics_metadata.loc[:, columns].copy()

df_genomics_metadata = df_genomics_metadata.sort_values(by=["#CHROM", "POS"], ascending=True)
df_rsid_map = df_genomics_metadata["RSID"].replace(":", ".")

df_genomics_metadata

### Load linkage disequilibrium matrices

In [ ]:
LD_method = "RH"
# List of RSIDs to check linkage disequilibrium
custom_rsids = [
    # "i_rs10762276:71052469:C:T",
    # "i_rs11400433:71147641:G:GA",
    # "i_rs142968171:71024743:AT:A",
    # "i_rs2229387:71151885:C:T",
    # "i_rs57252689:71147002:A:G",
    # "i_rs57985624:71151329:C:T",
    # "i_rs7089343:71165632:C:T",
    # "i_rs72166113:71098436:AGT:A",
    # "i_rs74138361:71099557:C:G",
    # "i_rs7904932:71150946:C:T",
]


rsids = df_genomics_metadata.index.tolist()
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(
    "REDS-Index",  # Always REDS-Index cohort
    f"Genomics-{GENE_ID}_Metadata",
    LD_method,
    "LinkageDisequilibrium",
    "ALL",
)
df_ld_matrix = load_dataframe_from_file(input_dirpath / input_filename, index_col=["SNP1", "SNP2"])
df_ld_matrix = df_ld_matrix.loc[pd.IndexSlice[rsids, rsids], :]
df_r2_thresh = df_ld_matrix[df_ld_matrix["r2"] >= 0]
r2_rsids = (
    df_r2_thresh.index.get_level_values("SNP1")
    .union(df_r2_thresh.index.get_level_values("SNP2"))
    .unique()
)
df_ld_matrix = df_ld_matrix.loc[pd.IndexSlice[r2_rsids, r2_rsids], :]
if custom_rsids:
    df_ld_matrix = df_ld_matrix.loc[pd.IndexSlice[custom_rsids, custom_rsids], :].copy()
df_ld_matrix

In [ ]:
chromosome = sorted(df_genomics_metadata["#CHROM"].unique())
ordered_positions = df_genomics_metadata.copy()
ordered_positions["POS"] = df_genomics_metadata["POS"]

chromosome_locx = "bottom"
chromosome_locy = "left"
figsize = (10, 10)
spacer = 0.5
cbar_dim = 0.25
chromosome_dim = 0.5
connector_color = ["xkcd:grey"]

snp_linewidth = 0.5
connector_linewidth = snp_linewidth / 2
snp_color = ["xkcd:green"]

label_fontdict = {"size": "large"}
label_fontproperties = {"weight": "bold"}
tick_fontdict = {"size": "medium"}
grid_kwargs = {"color": "xkcd:black", "linewidth": 0.01}
factor = 1e4

ethnicity = "ALL"
value_type = ["D'", "r2"]
ordered_positions = ordered_positions.loc[r2_rsids].sort_values(by=["#CHROM", "POS"])
ordered_positions


fig, axes = create_LD_heatmap_w_chromosom_mapping(
    df_ld_matrix,
    value_type=value_type,
    ordered_positions=ordered_positions,
    chromosome=chromosome,
    chromosome_locx=chromosome_locx,
    chromosome_locy=chromosome_locy,
    figsize=figsize,
    cmap=["Blues", "Reds"],
    dropna=False,
    snp_color="xkcd:green",
    diag_color="xkcd:black",
    nan_color="xkcd:dark grey",
    label_fontdict=label_fontdict,
    label_fontproperties=label_fontproperties,
    tick_fontdict=tick_fontdict,
    grid_kwargs=grid_kwargs,
)
ax_LDheatmap, ax_chromosome_x, ax_chromosome_y, ax_cbar_x, ax_cbar_y = axes